# 01 Data preparation

**Predictive Dairy Performance Analytics for Farm Consulting**  
D. Darren McGee — DSCI 8413 Week 2

This notebook documents how the Elanco lactation extract is assessed and prepared. Quality counts were produced in JMP on the local analysis table `lactations_with_farmid`. Raw farm rows are not stored in this repository.

Workflow: raw extract (Elanco) → cleaning rules → processed table (Elanco) → encoded summaries (GitHub).

## 1. Load

Set `YIELDGAP_DATA` to a local CSV/JMP export if you are running this on the Elanco machine. If the variable is unset, the notebook prints the JMP audit only.

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from preprocessing import JMP_AUDIT, FIT_START, FIT_END, HOLDOUT_START, HOLDOUT_END, HOLDOUT_ELIGIBLE_FSH_END

data_path = os.environ.get("YIELDGAP_DATA")
print("Local file:" if data_path else "No local file (JMP audit only).", data_path or "")

## 2. Inspect the raw extract

4,803,135 cow-lactation rows × 250 columns. Fresh dates 29 Aug 2023–29 Aug 2026. 320 farms after dropping missing `HerdCode`.

In [ ]:
for k, v in JMP_AUDIT.items():
    print(f"{k:24} {v}")

## 3–4. Quality problems and cleaning

- Drop `AccountName` and `HerdCode` from shared files; use `FarmID`.
- Exclude 758 bad `Lact` values from the gap calculation.
- 1,364 extra key rows: drop exact duplicates; keep true re-fresh events.
- 18.8% of rows have no `Milk1`–`Milk9`. Do not impute milk.
- 0 impossible milk/DIM slots. Keep 273 rows with milk > 250 and 3 with DIM > 800.
- Do not use `Milk at Test`, `DIM at Test`, 305-day ME, or lifetime yield in the expected-yield fit.
- Rank constraints from complete Y/N flags; treat missing diet/housing as data gaps.

## 5. Integration

No feed or shipment join this week. `FarmID` comes from a private 320-row crosswalk that is not in this repository.

## 6–7. Features and leakage

Expected milk uses DIM, lactation group, and month of that lactation’s fresh date.

- Fit window: 2023-08-29 to 2026-02-28
- Holdout window: 2026-03-01 to 2026-08-29
- Score only test-day slots that exist.
- Whole-lactation holdout check: fresh date through 2026-05-31 or at least two milk slots.
- Summarize holdout gaps by month of calving.

In [ ]:
print("Fit     ", FIT_START, "→", FIT_END)
print("Holdout ", HOLDOUT_START, "→", HOLDOUT_END)
print("Eligible FshDate for whole-lactation check ≤", HOLDOUT_ELIGIBLE_FSH_END)

## 8. Processed data

Processed tables remain in the Elanco workspace next to the raw extract. This repository stores rules only.